In [0]:
# Databricks notebook source

from __future__ import annotations

import time
import uuid
import requests
from datetime import datetime, timezone
from typing import Any

from pyspark.sql.types import StructField, StructType, StringType

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
dbutils.widgets.text("secret_scope", "phc-txdot-dev")
dbutils.widgets.text("secret_key", "txdot-app-token")
dbutils.widgets.text("closed_bids_url", "https://data.texas.gov/resource/de7b-7dna.json")
dbutils.widgets.text("closed_bids_source", "closed_bids")
dbutils.widgets.text("page_size", "50000")
dbutils.widgets.text("sleep_seconds", "0.5")
dbutils.widgets.text("request_timeout", "120")
dbutils.widgets.text("test_mode", "false")
dbutils.widgets.text("test_max_pages", "2")

catalog = dbutils.widgets.get("catalog")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
closed_bids_url = dbutils.widgets.get("closed_bids_url")
closed_bids_source = dbutils.widgets.get("closed_bids_source")
page_size = int(dbutils.widgets.get("page_size"))
sleep_seconds = float(dbutils.widgets.get("sleep_seconds"))
request_timeout = int(dbutils.widgets.get("request_timeout"))
test_mode = dbutils.widgets.get("test_mode").lower() == "true"
test_max_pages = int(dbutils.widgets.get("test_max_pages"))

token = dbutils.secrets.get(scope=secret_scope, key=secret_key)

BRONZE = f"{catalog}.bronze.closed_bids_raw"
WATERMARK = f"{catalog}.staging.audit_watermark"
LOG = f"{catalog}.staging.pipeline_run_log"

RUN_ID = str(uuid.uuid4())
SOURCE_NAME = closed_bids_source
BOOTSTRAP_START = "1900-01-01T00:00:00.000Z"

# ============================================================
# Mapping (matches Postgres exactly)
# ============================================================
RAW_COLS = [
    ":id",
    ":created_at",
    ":updated_at",
    ":version",
    "bid_code",
    "bid_item_description",
    "bid_item_quantity",
    "bid_item_sequence_number",
    "bid_item_unit_price_amount",
    "bid_rank_sequence_number",
    "bid_submit_sequence_number",
    "bid_total_amount",
    "control_section_job_csj",
    "controlling_project_id_ccsj",
    "county",
    "dbe_goal_percent",
    "district_division",
    "engineer_s_estimate_unit",
    "highway",
    "low_bidder_flag",
    "maximum_number_of_working",
    "measurement_unit",
    "project_actual_let_date",
    "project_classification",
    "project_estimated_let_date",
    "project_id",
    "project_limits_from",
    "project_limits_to",
    "project_name",
    "project_number",
    "project_type",
    "sealed_engineer_s_estimate",
    "sealed_engineer_s_estimate_1",
    "short_description",
    "spec_book_year",
    "specification_code",
    "state_project_number",
    "specification_description",
    "vendor_name",
    "federal_project_number",
    "force_account_amount",
    "force_account_description",
    "nbi_number",
    "utility_id",
    "alternative_bid_code",
    "a_b_engineer_s_estimate_amount",
]

COL_MAP = {c: c.lstrip(":").replace(":", "_") for c in RAW_COLS}
COL_MAP[":id"] = "socrata_id"
COL_MAP[":created_at"] = "socrata_created_at"
COL_MAP[":updated_at"] = "socrata_updated_at"
COL_MAP[":version"] = "socrata_version"

TARGET_COLS = [COL_MAP[c] for c in RAW_COLS] + [
    "ingestion_run_id",
    "ingested_at_utc",
]

BRONZE_SCHEMA = StructType([
    StructField(col_name, StringType(), True) for col_name in TARGET_COLS
])

# ============================================================
# Helpers
# ============================================================
def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def get_watermark() -> str | None:
    df = spark.sql(f"""
        SELECT watermark_value
        FROM {WATERMARK}
        WHERE source_name = '{sql_escape(SOURCE_NAME)}'
    """)
    rows = df.collect()
    return rows[0][0] if rows else None


def set_watermark(value: str) -> None:
    value_escaped = sql_escape(value)
    spark.sql(f"""
        MERGE INTO {WATERMARK} t
        USING (
            SELECT
                  '{sql_escape(SOURCE_NAME)}' AS source_name
                , '{value_escaped}' AS watermark_value
                , current_timestamp() AS updated_at_utc
        ) s
        ON t.source_name = s.source_name
        WHEN MATCHED THEN UPDATE SET
              t.watermark_value = s.watermark_value
            , t.updated_at_utc = s.updated_at_utc
        WHEN NOT MATCHED THEN INSERT (
              source_name
            , watermark_value
            , updated_at_utc
        ) VALUES (
              s.source_name
            , s.watermark_value
            , s.updated_at_utc
        )
    """)


def log_run(status: str, row_count: int, message: str) -> None:
    spark.sql(f"""
        INSERT INTO {LOG}
        VALUES (
            '{RUN_ID}',
            '10_ingest_closed_bids',
            '{sql_escape(status)}',
            {row_count},
            '{sql_escape(message)}',
            current_timestamp()
        )
    """)


# ============================================================
# API
# ============================================================
HEADERS = {
    "Accept": "application/json",
    "X-App-Token": token,
}


def fetch_page(offset: int, watermark: str | None) -> list[dict[str, Any]]:
    where_clause = f":updated_at > '{watermark or BOOTSTRAP_START}'"

    params = {
        "$select": ":*, *",
        "$where": where_clause,
        "$order": ":updated_at ASC, :id ASC",
        "$limit": page_size,
        "$offset": offset,
    }

    r = requests.get(
        closed_bids_url,
        headers=HEADERS,
        params=params,
        timeout=request_timeout,
    )

    if r.status_code != 200:
        raise Exception(f"API error {r.status_code}: {r.text[:1000]}")

    payload = r.json()
    if not isinstance(payload, list):
        raise Exception(f"Unexpected response format: {type(payload)}")

    return payload


# ============================================================
# Main
# ============================================================
watermark = get_watermark()
mode = "INCREMENTAL" if watermark else "BOOTSTRAP"

print("MODE:", mode)
print("WATERMARK:", watermark)

offset = 0
total = 0
max_updated_at = None
max_id_at_watermark = None
page = 0

try:
    while True:
        data = fetch_page(offset, watermark)

        if not data:
            break

        normalized = []
        for row in data:
            rec = {}

            for src_col in RAW_COLS:
                tgt_col = COL_MAP[src_col]
                val = row.get(src_col)
                rec[tgt_col] = None if val in (None, "") else str(val)

            rec["ingestion_run_id"] = RUN_ID
            rec["ingested_at_utc"] = utc_now()

            normalized.append(rec)

            ts = rec.get("socrata_updated_at")
            sid = rec.get("socrata_id")

            if ts and (max_updated_at is None or ts > max_updated_at):
                max_updated_at = ts
                max_id_at_watermark = sid
            elif ts and ts == max_updated_at and sid:
                if max_id_at_watermark is None or sid > max_id_at_watermark:
                    max_id_at_watermark = sid

        rows_for_df = [tuple(rec.get(col) for col in TARGET_COLS) for rec in normalized]
        df = spark.createDataFrame(rows_for_df, schema=BRONZE_SCHEMA)
        df.createOrReplaceTempView("incoming")

        update_clause = ",\n            ".join(
            [f"t.{col} = s.{col}" for col in TARGET_COLS if col != "socrata_id"]
        )
        insert_cols = ", ".join(TARGET_COLS)
        insert_vals = ", ".join([f"s.{c}" for c in TARGET_COLS])

        spark.sql(f"""
            MERGE INTO {BRONZE} t
            USING incoming s
            ON t.socrata_id = s.socrata_id
            WHEN MATCHED THEN UPDATE SET
                {update_clause}
            WHEN NOT MATCHED THEN INSERT (
                {insert_cols}
            ) VALUES (
                {insert_vals}
            )
        """)

        total += len(normalized)
        offset += page_size
        page += 1

        print(f"Loaded {total:,} rows")

        if test_mode and page >= test_max_pages:
            print("TEST MODE STOP")
            break

        time.sleep(sleep_seconds)

    if max_updated_at:
        set_watermark(max_updated_at)

    log_run("SUCCESS", total, f"{mode} complete. watermark={max_updated_at} id={max_id_at_watermark}")
    print("DONE:", total)

except Exception as exc:
    log_run("FAILED", total, str(exc))
    raise